# FSDP Multi-GPU Training on SageMaker

Train medical image segmentation models with Fully Sharded Data Parallel (FSDP) across multiple GPUs.

## Setup

In [ ]:
import sagemaker
from sagemaker.pytorch.estimator import PyTorch
import boto3

sagemaker_session = sagemaker.Session(boto3.Session(region_name='us-east-1'))



# sagemaker_session = sagemaker.Session()

region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()

print(f"Region: {region}")
print(f"Bucket: {bucket}")

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '../..'))
from utils import get_or_create_role

role = get_or_create_role()
print(f"SageMaker role: {role}")


## Data Path

Point to your S3 data location:

In [ ]:
bucket = 'YOUR_BUCKET_NAME'  # Replace with your S3 bucket name
data_path = f's3://{bucket}/segmentation_data/'
output_path = f's3://{bucket}/segmentation_data/output'

print(f"Training data: {data_path}")
print(f"Output path: {output_path}")

## Configure Training Job

In [ ]:
# Hyperparameters
hyperparameters = {
    'model_name': 'SegResNet',
    'batch_size': 2,
    'epochs': 10,
    'lr': 0.0001
}

# PyTorch Estimator with FSDP
estimator = PyTorch(
    entry_point='train_fsdp.py',
    source_dir='../code/training',
    role=role,
    instance_type='ml.g4dn.12xlarge',  # 4 GPUs
    instance_count=1,
    framework_version='2.0.0',
    py_version='py310',
    hyperparameters=hyperparameters,
    distribution={
        'pytorchddp': {
            'enabled': True
        }
    },
    keep_alive_period_in_seconds=600,
    disable_profiler=True,
    debugger_hook_config=False,
    sagemaker_session=sagemaker_session,
)

print("Estimator configured successfully")

## Start Training

In [ ]:
# Start training job
estimator.fit({'training': data_path}, wait=True, logs='All')

## Get Model Artifacts

In [ ]:
# Model artifacts location
model_data = estimator.model_data
print(f"Model artifacts: {model_data}")

## Training Metrics

View training metrics in CloudWatch or download TensorBoard logs from S3.

In [ ]:
# Get training job name
training_job_name = estimator.latest_training_job.name
print(f"Training job: {training_job_name}")

# CloudWatch logs
print(f"\nCloudWatch logs:")
print(f"https://console.aws.amazon.com/cloudwatch/home?region={region}#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs")